In [72]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent,
    tool,
)
from langgraph.graph import StateGraph, START, END

In [71]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
import pandas as pd
import io
from selenium.webdriver.chrome.options import Options
options = Options()
options.add_argument('--headless=new')
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)

In [69]:
from bs4 import SoupStrainer

In [67]:
import httpx

In [76]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [102]:
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
driver.get("https://navercomp.wisereport.co.kr/v2/company/c1010001.aspx?cmp_cd=001510")

In [38]:
from bs4 import BeautifulSoup
report = BeautifulSoup(driver.page_source).find_all('table', class_='gHead01 all-width')

In [39]:
report_text = "\n".join([pd.read_html(io.StringIO(str(x)))[0].to_json() for x in report])

In [40]:
report_text

'{"\\uc2e0\\uc6a9\\ub4f1\\uae09":{"0":"KIS","1":"KR","2":"NICE","3":null},"BOND":{"0":"A- [20260311]","1":"A- [20260313]","2":null,"3":null},"CP":{"0":"A2 [20251218]","1":"A2 [20251218]","2":"A2 [20251215]","3":null}}\n{"\\uc8fc\\uc694\\uc8fc\\uc8fc":{"0":"\\uc81c\\uc774\\uc564\\ub354\\ube14\\uc720\\ud30c\\ud2b8\\ub108\\uc2a4 \\uc678 1\\uc778 \\uc81c\\uc774\\uc564\\ub354\\ube14\\uc720\\ud30c\\ud2b8\\ub108\\uc2a4 \\uc678 1\\uc778","1":"\\uc790\\uc0ac\\uc8fc","2":null,"3":null},"\\ubcf4\\uc720\\uc8fc\\uc2dd\\uc218(\\ubcf4\\ud1b5)":{"0":92609439.0,"1":58683968.0,"2":null,"3":null},"\\ubcf4\\uc720\\uc9c0\\ubd84(%)":{"0":19.6,"1":12.42,"2":null,"3":null}}\n{"(\'\\uc8fc\\uc694\\uc7ac\\ubb34\\uc815\\ubcf4\', \'\\uc8fc\\uc694\\uc7ac\\ubb34\\uc815\\ubcf4\')":{"0":"\\ub9e4\\ucd9c\\uc561","1":"\\uc601\\uc5c5\\uc774\\uc775","2":"\\uc601\\uc5c5\\uc774\\uc775(\\ubc1c\\ud45c\\uae30\\uc900)","3":"\\uc138\\uc804\\uacc4\\uc18d\\uc0ac\\uc5c5\\uc774\\uc775","4":"\\ub2f9\\uae30\\uc21c\\uc774\\uc775","5":"\

In [41]:
llm  = ChatOpenAI(model='gpt-4o', temperature=0.2)
response = llm.invoke(f"제공된 데이터 분석해봐 {report_text}")

In [44]:
print(response.content)

제공된 데이터는 여러 금융 및 재무 지표를 포함하고 있으며, 이를 통해 특정 기업의 재무 상태와 성과를 분석할 수 있습니다. 주요 내용을 요약하면 다음과 같습니다:

1. **신용등급 및 채권(CP) 등급**:
   - 신용등급은 KIS, KR, NICE 등급을 보유하고 있으며, 채권 등급은 A-로 평가되었습니다.
   - CP(기업어음) 등급은 A2로 평가되었습니다.

2. **주요 주주 및 보유 주식**:
   - 주요 주주로는 '제이앤더블유파트너스 외 1인'과 '자사주'가 있으며, 각각 19.6%와 12.42%의 지분을 보유하고 있습니다.

3. **연간 재무 성과**:
   - 2022년부터 2024년까지의 매출액, 영업이익, 순이익 등의 변화를 볼 수 있습니다.
   - 2022년에는 매출액이 12,507억 원, 영업이익이 179억 원, 순이익이 94억 원이었으나, 2024년에는 매출액이 11,017억 원으로 감소하고, 영업이익과 순이익은 각각 -1,079억 원, -825억 원으로 적자를 기록했습니다.

4. **주요 재무 비율**:
   - ROE(자기자본이익률)는 2022년 1.52%에서 2024년 -13.9%로 감소했습니다.
   - PER(주가수익비율)은 2022년 31.26배에서 2023년 119.72배로 증가했으며, 2024년에는 데이터가 제공되지 않았습니다.
   - PBR(주가순자산비율)은 0.41배에서 0.35배로 소폭 감소했습니다.

5. **기타 재무 지표**:
   - 순부채비율은 2022년 176.48%에서 2024년 165.82%로 감소했습니다.
   - EV/EBITDA(기업가치/세전영업이익)는 2022년 33.05배에서 2023년 37.64배로 증가했으며, 2024년에는 -13.87배로 나타났습니다.

6. **분기별 재무 성과**:
   - 2025년 1분기부터 3분기까지의 매출액, 영업이익, 순이익 등의 변화를 볼 수 있으며, 분기별로 실적이 다소 변동하고 있습니다.

이 데이터는 기업의 재무 상태와 성과를 분석하는 데

In [52]:
@tool
def finance_report(company_code : str) -> str:
    """
    company_code : 회사 종목 코드
    종목코드 회사의 연간, 분기 재무제표를 값을 리턴하는 함수


    return : 연간, 분기 재무제표 값
    """
    options = Options()
    options.add_argument('--headless=new')
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    driver.get(f"https://navercomp.wisereport.co.kr/v2/company/c1010001.aspx?cmp_cd={company_code}")
    report = BeautifulSoup(driver.page_source).find_all('table', class_='gHead01 all-width')
    report_text = "\n".join([pd.read_html(io.StringIO(str(x)))[0].to_json() for x in report])
    return report_text


In [47]:
finance_report('005930')

'{"\\uc2e0\\uc6a9\\ub4f1\\uae09":{"0":"KIS","1":"KR","2":"NICE","3":null},"BOND":{"0":null,"1":null,"2":null,"3":null},"CP":{"0":null,"1":null,"2":null,"3":null}}\n{"\\uc8fc\\uc694\\uc8fc\\uc8fc":{"0":"\\uc0bc\\uc131\\uc0dd\\uba85\\ubcf4\\ud5d8 \\uc678 15\\uc778 \\uc0bc\\uc131\\uc0dd\\uba85\\ubcf4\\ud5d8 \\uc678 15\\uc778","1":"\\uad6d\\ubbfc\\uc5f0\\uae08\\uacf5\\ub2e8","2":"BlackRock Fund Advisors \\uc678 15\\uc778 BlackRock Fund Advisors \\uc678 15\\uc778","3":"\\uc790\\uc0ac\\uc8fc"},"\\ubcf4\\uc720\\uc8fc\\uc2dd\\uc218(\\ubcf4\\ud1b5)":{"0":1170128680,"1":458637667,"2":300391061,"3":120813769},"\\ubcf4\\uc720\\uc9c0\\ubd84(%)":{"0":19.77,"1":7.75,"2":5.07,"3":2.04}}\n{"(\'\\uc8fc\\uc694\\uc7ac\\ubb34\\uc815\\ubcf4\', \'\\uc8fc\\uc694\\uc7ac\\ubb34\\uc815\\ubcf4\')":{"0":"\\ub9e4\\ucd9c\\uc561","1":"\\uc601\\uc5c5\\uc774\\uc775","2":"\\uc601\\uc5c5\\uc774\\uc775(\\ubc1c\\ud45c\\uae30\\uc900)","3":"\\uc138\\uc804\\uacc4\\uc18d\\uc0ac\\uc5c5\\uc774\\uc775","4":"\\ub2f9\\uae30\\uc21c\

In [51]:
@tool
def get_code(company_name: str) -> dict:
    """
    사용자가 회사 이름을 전달하면 해당 회사의 종목 코드를 dict형태로 반환한다.
    """
    df = pd.read_csv(r"C:\Users\playdata2\OneDrive\Desktop\20260312_openapikey\data_2058_20260323.csv", encoding='cp949')
    return df[df['한글 종목명'].apply(lambda x : x.find(company_name) > -1)].to_json()

In [53]:
tools = [finance_report, get_code ]

In [54]:
prompt = ChatPromptTemplate([
    ( "system", """당신은 재무 분석가입니다. CFO 수준의 재무제표 전문가입니다.
     사용자가 제공한 재무제표 데이터를 기반으로 재무 건전성, 수익성, 성장성, 안정성을 종합 분석할 것
     그리고 재무 분석가의 의견으로 이야기할 것 """ ),
     ("human", "{input}"),
     ('placeholder', "{agent_scratchpad}")]
)


In [55]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [56]:
result = agent_executor.invoke({"input" : "광전자"})



> Entering new AgentExecutor chain...

Invoking: `get_code` with `{'company_name': '광전자'}`


{"\ud45c\uc900\ucf54\ub4dc":{"177":"KR7017900002"},"\ub2e8\ucd95\ucf54\ub4dc":{"177":"017900"},"\ud55c\uae00 \uc885\ubaa9\uba85":{"177":"\uad11\uc804\uc790\ubcf4\ud1b5\uc8fc"},"\ud55c\uae00 \uc885\ubaa9\uc57d\uba85":{"177":"\uad11\uc804\uc790"},"\uc601\ubb38 \uc885\ubaa9\uba85":{"177":"AUK"},"\uc0c1\uc7a5\uc77c":{"177":"1996\/10\/16"},"\uc2dc\uc7a5\uad6c\ubd84":{"177":"KOSPI"},"\uc99d\uad8c\uad6c\ubd84":{"177":"\uc8fc\uad8c"},"\uc18c\uc18d\ubd80":{"177":null},"\uc8fc\uc2dd\uc885\ub958":{"177":"\ubcf4\ud1b5\uc8fc"},"\uc561\uba74\uac00":{"177":"500"},"\uc0c1\uc7a5\uc8fc\uc2dd\uc218":{"177":57943763}}
Invoking: `finance_report` with `{'company_code': '017900'}`


{"\uc2e0\uc6a9\ub4f1\uae09":{"0":"KIS","1":"KR","2":"NICE","3":null},"BOND":{"0":null,"1":null,"2":null,"3":null},"CP":{"0":null,"1":null,"2":null,"3":null}}
{"\uc8fc\uc694\uc8fc\uc8fc":{"0":"Nakajima Hirokazu \uc678 6\uc778 Nakajima Hi

In [60]:
@tool
def get_news(company_name : str) -> str:
    """  
    company_name : 회사명
    회사명으로 뉴스를 검색하여 해당 내용을 반환한다.


    return : 회사명으로 검색된 뉴스의 총 텍스트값을 리턴
    """
    global total_naver_url
    client_id = "YXw4dmid0O2qTQNfhcY2"
    client_secret = "BksteLLT_U"
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }
    params = {'query' : company_name , 'display' : 100, 'start': 1, "sort" : "date"}
    response = httpx.get(url, headers=headers, params=params)
    total_naver_url = [url['link'] for url in response.json()['items'] if 'naver.com' in url['link']]
    bs4_kwargs = {
        'parse_only' : SoupStrainer("div", id="newsct_article")
    }
    rt = WebBaseLoader(total_naver_url[0],  bs_kwargs=bs4_kwargs)


    return " ".join([WebBaseLoader(x,bs_kwargs=bs4_kwargs).load()[0].page_content.strip() for x in total_naver_url])


In [61]:
prompt_news = ChatPromptTemplate([
    ( "system", """당신은 기업 분석가입니다. 제공된 뉴스를 바탕으로 현재 기업의 상태, 미래 전망,
     사업의 건전성등을 기업에 대한 모든 정보를 출력합니다.""" ),
     ("human", "{input}"),
     ('placeholder', "{agent_scratchpad}")]
)

In [77]:
news_tools = [get_news]
news_agent = create_tool_calling_agent(llm, news_tools, prompt_news)
news_agent_executor = AgentExecutor(agent=news_agent, tools=news_tools, verbose=True)
news_result = news_agent_executor.invoke({"input" : "광전자"})



> Entering new AgentExecutor chain...

Invoking: `get_news` with `{'company_name': '광전자'}`


24일 오후 15시 35분 SK증권우(001515)가 등락률 +29.96%로 상승률 1위로 마감했다. SK증권우는 장 중 2,288,735주가 거래되었으며 주가는 전 거래일 대비 1,630원 오른 7,070원에 마감했다.한편 SK증권우의 PER은 883.75로 매우 높은 수준을 보이고 있으며, 이는 시장에서 해당 종목에 대해 높은 평가를 내리고 있음을 시사한다. ROE는 N/A로 나타났다.이어 상승률 2위 광전자(017900)는 주가가 29.78% 폭등하며 종가 2,615원에 상승 마감했다. 상승률 3위 한국전자홀딩스(006200)의 주가는 906원으로 21.12% 폭등하며 강세를 보였다. 상승률 4위 부광약품(003000)은 17.72% 급등하며 7,110원에 마감했다. 상승률 5위 에이엔피(015260)는 15.56%의 상승세를 타고 종가 661원에 마감했다.6위 삼양패키징(272550)은 종가 14,040원으로 14.99% 상승 마감했다. 7위 세아베스틸지주(001430)는 종가 68,900원으로 14.07% 상승 마감했다. 8위 프레스티지바이오파마(950210)는 종가 10,350원으로 13.49% 상승 마감했다. 9위 엘앤에프(066970)는 종가 125,000원으로 11.51% 상승 마감했다. 10위 에코프로머티(450080)는 종가 68,000원으로 11.48% 상승 마감했다.



이밖에도 SGC에너지(005090) ▲10.65%, 코오롱인더(120110) ▲10.38%, LG에너지솔루션(373220) ▲10.25%, 비비안(002070) ▲9.77%, 엘브이엠씨홀딩스(900140) ▲9.71%, 에이프로젠바이오로직스(003060) ▲9.42%, 삼일씨엔에스(004440) ▲9.16%, 풀무원(017810) ▲8.89%, 동양2우B(001527) ▲8.43%, 영풍(000670)

In [78]:
from pydantic import BaseModel, Field

In [79]:
class CompanyAnalysisNews(BaseModel):
    current_status : str = Field(description="현재 기업의 상태 요약")
    future_outlook : str = Field(description="미래 전망 및 성장 가능성")
    business_health : str = Field(description="사업의 건정성 및 리스크 요인")
    core_keyword: list[str] = Field(description="기업과 관련된 핵심 뉴스 키워드")


structured_llm = llm.with_structured_output(CompanyAnalysisNews)
final_result = structured_llm.invoke(
    f"다음 기업 분석 텍스트를 분석하여 구조화된 데이터로 추출할 것\n\n {news_result['output']}"
)


In [80]:
final_result.model_dump_json()

'{"current_status":"광전자는 최근 주가가 29.78% 상승하며 주목받고 있습니다. 이는 시장에서의 긍정적인 평가와 투자자들의 관심을 반영하는 것으로 보입니다. 거래량은 945만 6440주, 거래대금은 231억 6800만 원, 시가총액은 1515억 원에 달합니다.","future_outlook":"광전자의 주가 상승은 긍정적인 시장 반응을 반영하지만, 이는 단기적인 투자자들의 관심에 의한 것일 수 있습니다. 장기적인 관점에서 기업의 지속 가능한 성장 가능성을 평가하기 위해서는 광전자의 기술 혁신, 시장 점유율, 경쟁력, 그리고 산업 내 위치에 대한 심층적인 분석이 필요합니다.","business_health":"광전자의 최근 주가 상승은 긍정적인 시장 반응을 나타내지만, 재무 지표와 구체적인 사업 성과에 대한 정보는 제공되지 않았습니다. 따라서, 기업의 재무 건전성이나 장기적인 수익성에 대한 판단은 추가적인 재무 보고서나 분석이 필요합니다.","core_keyword":["주가 상승","시장 평가","투자자 관심","재무 건전성","기술 혁신","시장 점유율","경쟁력","산업 내 위치"]}'

In [82]:
import json

In [84]:
stock_url = "https://m.stock.naver.com/front-api/external/chart/domestic/info?symbol=017900&requestType=1&startTime=20230614&endTime=20240513&timeframe=day"

In [85]:
data = eval(httpx.get(stock_url).text.strip())

In [86]:
stock_json = json.dumps(data)

In [87]:
rt = llm.invoke(f"데이터 분석 : {stock_json}")

In [88]:
rt

AIMessage(content='이 데이터는 주식 시장의 일일 거래 정보를 나타내고 있습니다. 각 행은 특정 날짜에 대한 정보를 포함하고 있으며, 각 열은 다음과 같은 정보를 제공합니다:\n\n1. 날짜 (예: "20230614")\n2. 시가 (시작 가격)\n3. 고가 (최고 가격)\n4. 저가 (최저 가격)\n5. 종가 (마감 가격)\n6. 거래량 (거래된 주식 수)\n7. 외국인 소진율 (외국인 투자자들이 보유한 주식의 비율)\n\n이 데이터를 분석하여 다음과 같은 인사이트를 얻을 수 있습니다:\n\n1. **가격 변동 분석**: 시가, 고가, 저가, 종가를 통해 주식의 일일 가격 변동을 분석할 수 있습니다. 예를 들어, 특정 날짜에 가격이 급등하거나 급락한 경우 그 이유를 분석할 수 있습니다.\n\n2. **거래량 분석**: 거래량을 통해 주식의 유동성을 평가할 수 있습니다. 거래량이 높은 날은 시장의 관심이 집중된 날일 가능성이 높습니다.\n\n3. **외국인 투자자 동향**: 외국인 소진율을 통해 외국인 투자자들의 투자 동향을 파악할 수 있습니다. 외국인 소진율이 높아지면 외국인 투자자들이 주식을 많이 보유하고 있다는 것을 의미합니다.\n\n4. **추세 분석**: 여러 날의 데이터를 종합하여 주식의 장기적인 추세를 분석할 수 있습니다. 예를 들어, 지속적인 가격 상승이나 하락 추세를 파악할 수 있습니다.\n\n5. **특정 이벤트의 영향**: 특정 날짜에 발생한 이벤트가 주식 가격에 미친 영향을 분석할 수 있습니다. 예를 들어, 기업 실적 발표나 경제 지표 발표가 주식 가격에 미친 영향을 평가할 수 있습니다.\n\n이러한 분석을 통해 투자 전략을 수립하거나 시장의 전반적인 동향을 파악하는 데 도움을 받을 수 있습니다. 추가적인 분석을 위해서는 통계적 방법이나 머신러닝 기법을 활용할 수도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completio

In [89]:
def get_data(company_code : str, sdate : str, edate : str) -> str:
    stock_url = f"https://m.stock.naver.com/front-api/external/chart/domestic/info?symbol={company_code}&requestType=1&startTime={sdate}&endTime={edate}&timeframe=day"
    data = eval(httpx.get(stock_url).text.strip())
    return json.dumps(data)


tmp = get_data('017900', '20260101', '20260325')


In [90]:
@tool
def get_data(company_code : str, sdate : str, edate : str) -> str:
    """
    company_code  : 종목코드 (예: 005930)
    sdate : 데이터 시작날짜 (예: 20260102)
    edate : 데이터 종료날짜 (예: 20260324)
    주식 데이터를 가져오는 함수
    return 종목의 시가, 고가, 저가,종가, 거래량 등의 데이터
    """
    stock_url = f"https://m.stock.naver.com/front-api/external/chart/domestic/info?symbol={company_code}&requestType=1&startTime={sdate}&endTime={edate}&timeframe=day"
    data = eval(httpx.get(stock_url).text.strip())
    return json.dumps(data)


In [91]:
stock_prompt = ChatPromptTemplate([
    ( "system", """당신은 20년 경력의 전업 투자가입니다. 주가의 움직임 속에 모든 정보와 투자자의 심리가 있다고 판단하고 있습니다. 주어진 데이터를 통해서 기술적 분석, 통계등을 이용해서 상태를 진단하고, 냉철하게 분석할 것
     사용자가 종목명을 입력하면 도구를 활용해서 종목 코드를 가져오고, 도구를 활용해서
     데이터를 가져올때 제시된 날짜부터 약 300일치 데이터를 가져올 것""" ),
     ("human", "{input}"),
     ('placeholder', "{agent_scratchpad}")]
)
stock_tools = [get_data, get_code ]
stock_agent = create_tool_calling_agent(llm, stock_tools, stock_prompt)
stock_agent_executor = AgentExecutor(agent=stock_agent, tools=stock_tools, verbose=True)
result = stock_agent_executor.invoke({"input" : "2026년 3월 25일 기준 광전자 주가 정보를 분석해봐"})




> Entering new AgentExecutor chain...

Invoking: `get_code` with `{'company_name': '광전자'}`


{"\ud45c\uc900\ucf54\ub4dc":{"177":"KR7017900002"},"\ub2e8\ucd95\ucf54\ub4dc":{"177":"017900"},"\ud55c\uae00 \uc885\ubaa9\uba85":{"177":"\uad11\uc804\uc790\ubcf4\ud1b5\uc8fc"},"\ud55c\uae00 \uc885\ubaa9\uc57d\uba85":{"177":"\uad11\uc804\uc790"},"\uc601\ubb38 \uc885\ubaa9\uba85":{"177":"AUK"},"\uc0c1\uc7a5\uc77c":{"177":"1996\/10\/16"},"\uc2dc\uc7a5\uad6c\ubd84":{"177":"KOSPI"},"\uc99d\uad8c\uad6c\ubd84":{"177":"\uc8fc\uad8c"},"\uc18c\uc18d\ubd80":{"177":null},"\uc8fc\uc2dd\uc885\ub958":{"177":"\ubcf4\ud1b5\uc8fc"},"\uc561\uba74\uac00":{"177":"500"},"\uc0c1\uc7a5\uc8fc\uc2dd\uc218":{"177":57943763}}
Invoking: `get_data` with `{'company_code': '017900', 'sdate': '20250324', 'edate': '20260324'}`


[["\ub0a0\uc9dc", "\uc2dc\uac00", "\uace0\uac00", "\uc800\uac00", "\uc885\uac00", "\uac70\ub798\ub7c9", "\uc678\uad6d\uc778\uc18c\uc9c4\uc728"], ["20250324", 1735, 1747, 1726, 1739, 22904, 18.06], ["2

In [92]:
print(result['output'])

2026년 3월 25일 기준으로 광전자의 주가는 다음과 같은 특징을 보입니다:

1. **종가**: 2615원
2. **시가**: 2055원
3. **고가**: 2615원
4. **저가**: 2055원
5. **거래량**: 9,665,572주

### 기술적 분석
- **가격 변동성**: 이 날은 주가가 크게 상승하여 고가와 저가의 차이가 상당히 큽니다. 이는 투자자들의 강한 매수세가 있었음을 나타냅니다.
- **거래량**: 거래량이 매우 높아, 시장의 관심이 집중된 날임을 알 수 있습니다. 이는 주가의 급격한 변동과 관련이 있을 수 있습니다.
- **추세**: 최근 며칠간 주가는 상승세를 보이고 있으며, 특히 3월 24일과 25일 사이에 급격한 상승이 있었습니다.

### 투자자 심리
- **강한 매수세**: 주가의 급등과 높은 거래량은 투자자들이 적극적으로 매수에 나섰음을 시사합니다.
- **시장 기대감**: 주가의 급등은 긍정적인 뉴스나 기대감이 반영되었을 가능성이 큽니다.

### 결론
광전자의 주가는 최근 며칠간 강한 상승세를 보이고 있으며, 특히 3월 25일에는 큰 폭으로 상승했습니다. 이는 투자자들의 강한 매수세와 시장의 긍정적인 기대감이 반영된 결과로 보입니다. 이러한 급등세가 지속될지 여부는 추가적인 시장 분석과 뉴스 흐름을 통해 판단해야 할 것입니다.


In [93]:
from typing import TypedDict
import asyncio

In [94]:
class CompanyState(TypedDict):
    question : str
    company_finance : str
    company_news : str
    company_stock : str
    final_report : str

In [95]:
async def finance_node(state : CompanyState):
    """
    사용자 입력한 회사의 재무제표 정보를 가져와서 분석하여 리턴하는 노드
    """
    prompt = ChatPromptTemplate([
                ( "system", """당신은 금융 감독위원회의 재무 분석가입니다. 검찰 수준으로 검토할 것
                사용자가 제공한 재무제표 데이터를 기반으로 재무 건전성, 수익성, 성장성, 안정성을 종합 분석할 것
                연간, 분기별로 별도로 분석해
                그리고 재무 분석가의 의견으로 이야기할 것 """ ),
                ("human", "{input}"),
                ('placeholder', "{agent_scratchpad}")]
            )




    tools = [finance_report, get_code ]


    agent = create_tool_calling_agent(llm, tools, prompt)
    agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


    result = agent_executor.ainvoke({"input" : state['question'] })
    return {'company_finance' : result['output']}

In [96]:
async def news_node(state : CompanyState):
    """
    사용자 입력한 회사의 뉴스 정보를 분석하여 리턴하는 노드
    """
    prompt_news = ChatPromptTemplate([
    ( "system", """당신은 기업 분석가입니다. 제공된 뉴스를 바탕으로 현재 기업의 상태, 미래 전망,
     사업의 건전성등을 기업에 대한 모든 정보를 출력합니다.""" ),
     ("human", "{input}"),
     ('placeholder', "{agent_scratchpad}")]
)
    news_tools = [get_news]
    news_agent = create_tool_calling_agent(llm, news_tools, prompt_news)
    news_agent_executor = AgentExecutor(agent=news_agent, tools=news_tools, verbose=True)
    result = news_agent_executor.ainvoke({"input" : state['question'] })
    return {'company_news' : result['output']}


async def stock_node(state : CompanyState):
    """
    사용자 입력한 회사 주가정보를 분석하여 리턴하는 노드
    """
    stock_prompt = ChatPromptTemplate([
    ( "system", """당신은 20년 경력의 전업 투자가입니다. 주가의 움직임 속에 모든 정보와 투자자의 심리가 있다고 판단하고 있습니다. 주어진 데이터를 통해서 기술적 분석, 통계등을 이용해서 상태를 진단하고, 냉철하게 분석할 것
     사용자가 종목명을 입력하면 도구를 활용해서 종목 코드를 가져오고, 도구를 활용해서
     데이터를 가져올때 제시된 날짜부터 약 300일치 데이터를 가져올 것""" ),
     ("human", "{input}"),
     ('placeholder', "{agent_scratchpad}")]
)
    stock_tools = [get_data, get_code ]
    stock_agent = create_tool_calling_agent(llm, stock_tools, stock_prompt)
    stock_agent_executor = AgentExecutor(agent=stock_agent, tools=stock_tools, verbose=True)
    result = stock_agent_executor.invoke({"input" : state['question']})
    return {'company_stock' : result['output']}




async def summarize_node(state : CompanyState):
    """에이전트 전달한 자료를 기반으로 최종 종목에서 판단하는 노드"""
    prompt = ChatPromptTemplate.from_messages([
        ('system', """
        에이전트들이 정리한 자료를 바탕으로 최종 종목에 대해서 매수의견을 작성하세요
         - 기업에 대한 재무제표 정리: {company_finance}
         - 기업에 대한 뉴스정리 : {company_news}
         - 기업에 대한 주가정보 정리 : {company_stock}
       
         위의 제공된 정보를 바탕으로 최종 의견을 제시할 것
        """),
        ('human', "{question}")
    ])
    chain = prompt | llm
    result = await chain.ainvoke( {
        'company_finance' : state.get('company_finance', ""),
        'company_news' : state.get('company_news', ""),
        'company_stock' : state.get('company_stock', ""),
        'question' : state['question']}
    )


    return {'final_report' : result.content}


In [98]:
workflow = StateGraph(CompanyState)
workflow.add_node('finance_node', finance_node)
workflow.add_node('news_node', news_node)
workflow.add_node('stock_node', stock_node)
workflow.add_node('summarize_node', summarize_node)
workflow.add_edge(START, 'finance_node')
workflow.add_edge(START, 'news_node')
workflow.add_edge(START, 'stock_node')
workflow.add_edge('finance_node', 'summarize_node')
workflow.add_edge('news_node', 'summarize_node')
workflow.add_edge('stock_node', 'summarize_node')
workflow.add_edge('summarize_node', END)
app = workflow.compile()
png = app.get_graph().draw_mermaid_png()


In [101]:
with open("./stock_graph.png", "wb") as f:
    f.write(png)
#result = await app.ainvoke({'question' : "광전자에 대해서 2026년 3월 25일 기준으로 분석해서 알려줘"})
result = await news_agent_executor.ainvoke({
    "input": "광전자에 대해서 2026년 3월 25일 기준으로 분석해서 알려줘"
})

print(result['output'])



> Entering new AgentExecutor chain...

Invoking: `get_news` with `{'company_name': '광전자'}`


코스피 거래량 상위 종목들이 전반적으로 엇갈린 흐름을 보이고 있다. 25일 한국거래소에 따르면 흥아해운(003280)이 6800만주 이상 거래되며 코스피 종목 중 실시간 거래량 1위를 차지하고 있다. 현재 주가는 3490원이며, 시가총액의 약 28.50%에 해당하는 2389억 9200만원의 거래대금이 집중되고 있다. PER은 27.48, ROE는 12.66으로 안정적인 재무 지표를 보인다. SK증권(001510)은 2220원으로 주가가 하락하며 거래량 5700만주를 기록하고 있다. 거래대금은 1331억 700만원으로 시가총액의 약 12.70%에 달하며, PER 277.50, ROE -13.91로 상대적으로 높은 밸류에이션을 보여준다.광전자(017900)는 29.83% 폭등하며 현재가 3395원에 거래되고 있으며, 거래량 3100만주를 기록하고 있다. 대우건설(047040)은 1만 6430원, 대한해운(005880)은 2445원에 각각 거래되고 있으며 등락률은 소폭 상승세를 보인다. 부광약품(003000)은 29.96%의 상한가를 기록하며 주가 9240원에 거래되고 있다. 서울식품(004410), KEC(092220), 한국ANKOR유전(152550)은 각각 하락세를 보이며, 삼성전자(005930)는 1.21% 상승한 19만 2000원에 거래되고 있다.



한편 거래량 상위 20위권 종목들은 한국전자홀딩스(006200) ▲15.78%, 신성이엔지(011930) ▲3.96%, SK증권우(001515) ▲14.00%, 진흥기업(002780) ▲0.30%, 한화생명(088350) ▲3.10%, 경보제약(214390) ▲13.51%, 에이프로젠(007460) ▼2.69%, 한온시스템(018880) ▲3.18%, 인스코비(006490) ▲17.11%, 삼양패키징(272550) ▲18.23% 등의 성적을